In [ ]:
import kagglehub
import pandas as pd #to read files
import os
import matplotlib.pyplot as plt
from sklearn.preprocessing import OneHotEncoder
# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q1-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
# Load the CSV file
df = pd.read_csv(os.path.join(path, "Q1_data.csv"))

In [ ]:
# Task 2: Write your code here:
df.head() #display first 5 rows

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 5: Write your code here:
def check_target_distribution(df, target_column):
  df[target_column].hist(bins=30, edgecolor='black')

  plt.title(f"Target Distribution ({target_column})")
  plt.xlabel(target_column)
  plt.ylabel("Frequency")
  plt.grid(False)

  plt.show()

check_target_distribution(df, "Delivery_Time")

In [ ]:
# Task 1: Write your code here:
df2 = df.copy()
df2 = df2.drop('Order_ID',axis=1)

In [ ]:
df2

In [ ]:
# Task 2: Write your code here:
def check_missing_values(df2):
  missing_values = df2.isnull().sum()
  print("Missing Values per Column:")
  print(missing_values[missing_values > 0])
  if missing_values.any():
    print("\nHandle Missing Values as needed.")
  else:
    print("\nNo Missing Values Found.")

check_missing_values(df2)


df_dropna = df2.dropna()


In [ ]:
check_missing_values(df_dropna)

In [ ]:
# Task 3: Write your code here:
def check_duplicates(df):

  #TODO: get duplicated data using pandas
  duplicates = df.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_dropna)

In [ ]:
df_dropna

In [ ]:
# Task 4: Write your code here:
categorical_cols = df_dropna.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))




In [ ]:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_dropna[col] = le.fit_transform(df_dropna[col])
  label_encoders[col] = le

df_dropna

In [ ]:
# Task 5: Write your code here:
from sklearn.preprocessing import StandardScaler

features = df_dropna.columns.drop("Delivery_Time")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_dropna[features] = scaler.fit_transform(df_dropna[features])
df_dropna.head()

In [ ]:
# Task 6: Write your code here:
import seaborn as sns
def check_target_imbalance(df_dropna, target_column):
  print("Target Distribution:")
  print(df_dropna[target_column].value_counts(normalize=True))
  sns.countplot(x=df_dropna[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_dropna, "Delivery_Time")

#i believe the target is impalanced

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
import numpy as np

In [ ]:
# Task 1: Write your code here:
X = df_dropna.drop("Delivery_Time", axis=1).astype(float)
y = df_dropna['Delivery_Time'].astype(float)

In [ ]:
# Task 2,3,4,5: Write your code here:
n_splits = 5  # K=5 Folds
#model.fit(X_train, y_train) # train
#y_pred = model.predict(X_test) # validate
rf = RandomForestRegressor(n_estimators=100, max_depth=5, random_state=42)
lr_mae = []
# 5-Fold Cross-Validation, shuffled
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(kf.split(X)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")

  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]

  # Train

  rf.fit(X_train, y_train)

  # Validate
  y_pred = rf.predict(X_test.values)

  # Calculate evaluation metrics
  mae = mean_absolute_error(y_test, y_pred)
  # Store results
  lr_mae.append(mae)
average_losses = np.mean(lr_mae, axis=0)

plt.figure(figsize=(10, 6))
plt.plot(average_losses, label='Average Loss')
plt.xlabel('Iteration')
plt.ylabel('MSE Loss')
plt.title('Linear Regression Training Loss (Averaged Across Folds)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Task 1: Write your code here:
RandomForestRegressor_model = rf
RandomForestRegressor_importance = list(zip(X.columns, RandomForestRegressor_model.feature_importances_))
sorted_catboost_importance = sorted(RandomForestRegressor_importance, key=lambda x: x[1], reverse=True)

# Extract features and their importances
features, importances = zip(*RandomForestRegressor_importance)

# Plot feature importances
plt.figure(figsize=(18, 14))
plt.barh(features, importances, color='orange')
plt.xlabel('Importance Score')
plt.ylabel('Features')
plt.title('CatBoost Feature Importance')
plt.gca().invert_yaxis()  # Invert y-axis to show the most important features at the top
plt.show()

In [ ]:
# Task 2: Write your code here:


In [ ]:
# Task Bonus: Write your code here: